# TRPG Agent Inputs Notebook

这个 notebook 参考 `trpg_direct_play.ipynb` 的直接游玩流程，但侧重点不同：

- 每一回合优先展示 `Dicer / NPC Manager / Narrator / Director` 实际拿到的输入材料
- 使用 `engine.trace_turn(...)` 生成完整调试快照
- 展示完各 Agent 输入后，直接把这次 trace 的结果提交进 `engine.state`
- 这样既能连续游玩，也能清楚看到每个 Agent 本轮到底看到了什么

说明：运行开场或回合时，会真实调用当前配置的文本模型。

In [ ]:
from __future__ import annotations

import importlib
import json
import sys
from datetime import datetime
from pathlib import Path
from typing import Any

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if not (candidate / 'Code').exists():
            continue
        if (candidate / 'Story').exists() or (candidate / 'story').exists() or (candidate / 'Prompt').exists():
            return candidate
    raise RuntimeError(f'Unable to locate repo root from: {start}')


ROOT = find_repo_root(Path.cwd().resolve())
CODE_DIR = (ROOT / 'Code').resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import trpg_runtime
import trpg_runtime.engine as trpg_runtime_engine
import trpg_runtime.models as trpg_runtime_models
import trpg_runtime.prompt_loader as trpg_runtime_prompt_loader
import trpg_runtime.state_store as trpg_runtime_state_store

importlib.reload(trpg_runtime_models)
importlib.reload(trpg_runtime_prompt_loader)
importlib.reload(trpg_runtime_state_store)
importlib.reload(trpg_runtime_engine)
importlib.reload(trpg_runtime)

from trpg_runtime import MinimalTRPGEngine, PromptRepository, RuntimeLogEvent, RuntimeOptions, TurnDebugTrace, parse_player_action
from trpg_runtime.state_store import advance_clock, append_recent_events, apply_delta


def to_plain_data(value: Any) -> Any:
    if hasattr(value, 'model_dump'):
        return to_plain_data(value.model_dump(mode='python'))
    if isinstance(value, dict):
        return {key: to_plain_data(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_plain_data(item) for item in value]
    return value


def print_heading(title: str) -> None:
    print(f"\n{'=' * 18} {title} {'=' * 18}")


def print_json_block(title: str, value: Any) -> None:
    print_heading(title)
    print(json.dumps(to_plain_data(value), ensure_ascii=False, indent=2))


def print_text_block(title: str, text: str) -> None:
    print_heading(title)
    print(text)


def print_log_event(event: RuntimeLogEvent) -> None:
    print(f'[LOG][{event.phase}/{event.stage}] {event.message}')


def summarize_value(value: Any) -> str:
    value = to_plain_data(value)
    if isinstance(value, str):
        return f'str, {len(value)} chars'
    if isinstance(value, list):
        return f'list, {len(value)} items'
    if isinstance(value, dict):
        return f'dict, {len(value)} keys'
    return type(value).__name__


def print_context_overview(title: str, context: dict[str, Any]) -> None:
    print_heading(f'{title} - 输入概览')
    for key, value in context.items():
        print(f'- {key}: {summarize_value(value)}')


def print_agent_input(title: str, context: dict[str, Any], *, full_json: bool = True) -> None:
    print_context_overview(title, context)
    if full_json:
        print_json_block(f'{title} - 完整输入 JSON', context)


def summarize_state(state: Any) -> dict[str, Any]:
    state = to_plain_data(state)
    return {
        'turn_id': state['core']['meta']['turn_id'],
        'time': {
            'day': state['core']['meta']['game_day'],
            'hour': state['core']['meta']['game_hour'],
            'minute': state['core']['meta']['game_minute'],
        },
        'scene': {
            'location': state['core']['scene']['location'],
            'visible_npcs': state['core']['scene']['visible_npcs'],
            'interactive_objects': state['core']['scene']['interactive_objects'],
            'hazards': state['core']['scene']['hazards'],
        },
        'recent_events': state['core']['recent_events'],
        'director': state['agent_runtime']['director'],
    }


def sanitize_delta_operations(delta_ops: Any) -> list[Any]:
    sanitized = []
    for op in delta_ops:
        path = getattr(op, 'path', '') or ''
        value = getattr(op, 'value', None)
        if path.endswith('.current_task') and isinstance(value, str):
            op = op.model_copy(
                update={
                    'value': {
                        'summary': value,
                        'status': 'active',
                    }
                }
            )
        sanitized.append(op)
    return sanitized


def trace_turn_with_sanitized_deltas(engine: MinimalTRPGEngine, player_text: str, *, include_next_director: bool = True):
    with engine._state_lock:
        state_before = engine.state.model_copy(deep=True)

    action = parse_player_action(player_text)
    director_state_used = state_before.director.model_copy(deep=True)

    dicer_context = engine.build_dicer_context(action, state_before)
    npc_context = engine.build_npc_context(action, state_before)

    dicer_future = engine._executor.submit(engine._call_dicer_from_context, dicer_context)
    npc_future = engine._executor.submit(engine._call_npc_manager_from_context, npc_context)
    dicer_result = dicer_future.result()
    npc_result = npc_future.result()

    dicer_result = dicer_result.model_copy(update={'state_delta': sanitize_delta_operations(dicer_result.state_delta)})
    npc_result = npc_result.model_copy(update={'state_delta': sanitize_delta_operations(npc_result.state_delta)})

    state_after_dicer = apply_delta(state_before, dicer_result.state_delta)
    state_after_npc = apply_delta(state_after_dicer, npc_result.state_delta)

    current_turn_events = engine._collect_current_turn_events(action, dicer_result, npc_result)
    state_after_turn = append_recent_events(
        state_after_npc,
        current_turn_events,
        max_recent_events=engine.options.max_recent_events,
    )
    updated_meta = advance_clock(state_after_turn.meta, engine.options.minutes_per_turn)
    updated_meta = updated_meta.model_copy(update={'turn_id': state_after_turn.meta.turn_id + 1})
    state_after_turn = state_after_turn.model_copy(
        update={
            'core': state_after_turn.core.model_copy(update={'meta': updated_meta}),
            'agent_runtime': state_after_turn.agent_runtime.model_copy(
                update={'last_player_action_text': player_text}
            ),
        }
    )

    narrator_context = engine.build_narrator_context(
        action=action,
        dicer_result=dicer_result,
        npc_result=npc_result,
        director_state=director_state_used,
        state=state_after_turn,
    )
    narration = engine._call_narrator_from_context(narrator_context)
    state_after_turn = state_after_turn.model_copy(
        update={
            'agent_runtime': state_after_turn.agent_runtime.model_copy(
                update={'last_narration': narration}
            )
        }
    )
    state_after_turn = engine._record_turn_artifacts(
        state_after_turn,
        action=action,
        narration=narration,
        dicer_result=dicer_result,
        npc_result=npc_result,
        current_turn_events=current_turn_events,
        max_dialogue_window=engine.options.max_dialogue_window,
    )

    next_director_context = engine.build_next_director_context(
        action=action,
        dicer_result=dicer_result,
        npc_result=npc_result,
        narration=narration,
        state=state_after_turn,
    )
    next_director_result = None
    state_after_next_director = None
    if include_next_director:
        next_director_result = engine._call_director_from_context(next_director_context)
        next_director_result = next_director_result.model_copy(
            update={'state_delta': sanitize_delta_operations(next_director_result.state_delta)}
        )
        state_after_next_director = engine._apply_director_result_to_state(
            state_after_turn,
            next_director_result,
        )

    return TurnDebugTrace(
        action=action,
        director_state_used=director_state_used,
        dicer_context=dicer_context,
        dicer_result=dicer_result,
        state_before_turn=engine._normalize_game_state_instance(state_before),
        state_after_dicer=engine._normalize_game_state_instance(state_after_dicer),
        npc_context=npc_context,
        npc_result=npc_result,
        state_after_npc=engine._normalize_game_state_instance(state_after_npc),
        narrator_context=narrator_context,
        narration=narration,
        state_after_turn=engine._normalize_game_state_instance(state_after_turn),
        next_director_context=next_director_context,
        next_director_result=next_director_result,
        state_after_next_director=engine._normalize_game_state_instance(state_after_next_director),
    )


def apply_trace_to_engine(engine: MinimalTRPGEngine, trace) -> None:
    engine.state = trace.state_after_next_director or trace.state_after_turn
    if trace.next_director_context is not None:
        engine._last_director_context = trace.next_director_context
    if trace.next_director_result is not None:
        engine._last_director_result = trace.next_director_result
        engine._unread_director_result = trace.next_director_result


def save_turn_materials(root: Path, rule_code: str, story_code: str, turn_logs: list[dict[str, Any]]) -> Path:
    out_dir = root / 'outputs' / 'agent_inputs'
    out_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = out_dir / f'agent_inputs_{rule_code}_{story_code}_{stamp}.json'
    path.write_text(json.dumps(to_plain_data(turn_logs), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return path


def inspect_and_commit_turn(engine: MinimalTRPGEngine, player_text: str, turn_index: int, *, full_json: bool = True):
    print('[LOG][turn/parse_action] [Turn] Parsing player action...')
    print('[LOG][turn/build_contexts] [Turn] Building Dicer and NPC Manager contexts...')
    print('[LOG][turn/run_dicer_npc] [Turn] Running Dicer and NPC Manager...')
    print('[LOG][turn/apply_deltas] [Turn] Applying state deltas...')
    print('[LOG][turn/run_narrator] [Turn] Running Narrator...')
    print('[LOG][turn/run_director] [Turn] Running Director...')
    trace = trace_turn_with_sanitized_deltas(
        engine,
        player_text,
        include_next_director=True,
    )

    print_json_block('Parsed action', trace.action)
    print_json_block('State before turn - 摘要', summarize_state(trace.state_before_turn))
    print_json_block('Director state used by Narrator', trace.director_state_used)

    print_agent_input('Dicer', trace.dicer_context, full_json=full_json)
    print_agent_input('NPC Manager', trace.npc_context, full_json=full_json)
    print_agent_input('Narrator', trace.narrator_context, full_json=full_json)
    print_agent_input('Director (for next turn)', trace.next_director_context, full_json=full_json)

    print_json_block('Dicer output', trace.dicer_result)
    print_json_block('NPC Manager output', trace.npc_result)
    print_text_block('Narrator output', trace.narration)
    print_json_block('Director output (next turn)', trace.next_director_result)

    apply_trace_to_engine(engine, trace)
    print_json_block('Committed engine.state - 摘要', summarize_state(engine.state))

    return {
        'turn_index': turn_index,
        'player_text': player_text,
        'trace': to_plain_data(trace),
        'committed_state_summary': summarize_state(engine.state),
    }


## 默认配置

如果你想切规则、剧本、玩家名或回合上限，直接改下一格。

In [ ]:
RULE_CODE = 'DET'
STORY_CODE = 'THE_FIRSTMURDER'
PLAYER_NAME = '调查员'
MAX_TURNS = 100
SHOW_FULL_JSON = True

OPTIONS = RuntimeOptions(
    full_rule_text_for_agents=True,
    full_story_text_for_agents=True,
    dicer_dialogue_window=2,
    npc_dialogue_window=3,
    narrator_dialogue_window=5,
    director_dialogue_window=5,
    max_dialogue_window=5,
)

print({
    'RULE_CODE': RULE_CODE,
    'STORY_CODE': STORY_CODE,
    'PLAYER_NAME': PLAYER_NAME,
    'MAX_TURNS': MAX_TURNS,
    'SHOW_FULL_JSON': SHOW_FULL_JSON,
})


## 创建引擎并生成开场

这一格会走完整初始化流程：读取规则/剧本/状态文件，必要时自动解析 seed，然后生成开场。

In [ ]:
try:
    engine.shutdown(wait=False)
except Exception:
    pass

repo = PromptRepository()
engine = MinimalTRPGEngine.from_prompt_files(
    rule_code=RULE_CODE,
    story_code=STORY_CODE,
    player_name=PLAYER_NAME,
    prompt_repository=repo,
    options=OPTIONS,
)

opening = engine.generate_opening(event_callback=print_log_event)
display(Markdown('### Opening'))
print(opening)
print_json_block('Initial engine.state - 摘要', summarize_state(engine.state))
print('\n可用命令: /state, /director, /opening, /quit')


## 单回合示例

如果你只想先看一次材料流转，可以先改下面这格的 `PLAYER_INPUT`，然后运行它。

In [ ]:
PLAYER_INPUT = '我先检查书桌、尸体右手和散落的纸片。'
preview_log = inspect_and_commit_turn(engine, PLAYER_INPUT, turn_index=1, full_json=SHOW_FULL_JSON)
preview_log.keys()


## 连续游玩，但每回合先展示 Agent 输入

这一格参考 `trpg_direct_play.ipynb` 的交互方式，但每回合会先：

1. `trace_turn()` 生成完整快照
2. 展示四个 Agent 的输入材料
3. 展示本回合输出
4. 直接把 trace 结果提交回 `engine.state`

退出时会把每回合材料导出到 `outputs/agent_inputs/*.json`。

In [ ]:
turn_index = 1
turn_logs = []
saved_path = None

while turn_index <= MAX_TURNS:
    player_text = input(f'[{turn_index}/{MAX_TURNS}] 你要说什么？ ').strip()
    if not player_text:
        print('请输入内容。')
        continue

    if player_text == '/quit':
        if turn_logs:
            saved_path = save_turn_materials(ROOT, RULE_CODE, STORY_CODE, turn_logs)
            print(f'游戏循环已结束，材料记录已保存到: {saved_path}')
        else:
            print('游戏循环已结束，本次没有可保存的回合记录。')
        break

    if player_text == '/state':
        print_json_block('Current engine.state', engine.state)
        continue

    if player_text == '/director':
        print_json_block('Current DirectorState', engine.state.director)
        continue

    if player_text == '/opening':
        print_text_block('Opening', opening)
        continue

    log_item = inspect_and_commit_turn(engine, player_text, turn_index=turn_index, full_json=SHOW_FULL_JSON)
    turn_logs.append(log_item)
    turn_index += 1

if turn_index > MAX_TURNS:
    if turn_logs:
        saved_path = save_turn_materials(ROOT, RULE_CODE, STORY_CODE, turn_logs)
        print(f'已达到最大回合数，材料记录已保存到: {saved_path}')
    else:
        print('已达到最大回合数。')
